In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os
os.chdir('/Users/eshitarajvegesna/crypto')  # set working directory


df = pd.read_parquet("data/processed/features.parquet")
print(df.shape)
df.head()

In [ ]:
df.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df["future_vol"].dropna().hist(bins=100, ax=axes[0])
axes[0].set_title("Distribution of future_vol (60s rolling std)")
axes[0].set_xlabel("future_vol")

pcts = np.arange(1, 100)
vals = np.nanpercentile(df["future_vol"].dropna(), pcts)
axes[1].plot(pcts, vals)
axes[1].set_xlabel("Percentile")
axes[1].set_ylabel("future_vol")
axes[1].set_title("Percentile plot — choose threshold at 85th-90th")

tau = np.nanpercentile(df["future_vol"].dropna(), 85)
axes[1].axhline(tau, color="red", linestyle="--", label=f"85th = {tau:.6f}")
axes[1].legend()
plt.tight_layout()
Path("reports").mkdir(exist_ok=True)
plt.savefig("reports/future_vol_dist.png", dpi=120)
plt.show()

print(f"Recommended threshold tau = {tau:.6f}")
print(f"Spike rate at tau: {(df['future_vol'] >= tau).mean():.1%}")

In [ ]:
feat_cols = [c for c in df.columns 
              if c not in ["ts", "product_id", "future_vol", "label"]
              and df[c].dtype in ["float64", "float32", "int64"]]

corr = df[feat_cols + ["future_vol"]].corr()["future_vol"].drop("future_vol").sort_values()

fig, ax = plt.subplots(figsize=(10, 8))
corr.plot(kind="barh", ax=ax)
ax.set_title("Feature correlation with future_vol")
plt.tight_layout()
plt.savefig("reports/feature_correlation.png", dpi=120)
plt.show()

In [ ]:
df_sorted = df.sort_values("ts")
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(pd.to_datetime(df_sorted["ts"], unit="s"), 
        df_sorted["spread_bps"], alpha=0.5, linewidth=0.5)
ax.set_title("Bid-Ask Spread (bps) over time")
plt.tight_layout()
plt.savefig("reports/spread_over_time.png", dpi=120)
plt.show()

In [ ]:
tau = np.nanpercentile(df["future_vol"].dropna(), 85)
print(f"Set spike_threshold: {tau:.6f} in config.yaml")
print(f"Spike rate: {(df['future_vol'] >= tau).mean():.1%}")